# SafeLoRA for Quantized Models - Exploration Notebook

This notebook provides a step-by-step exploration of:
1. Loading quantized models (4-bit, 8-bit)
2. Understanding the SafeLoRA projection mechanism
3. Applying SafeLoRA to restore safety alignment
4. Evaluating safety before and after projection

**ECE 285 Project - Safety Alignment of Quantized Models**


## Setup


In [ ]:
import os
import sys

# Add project root to path
project_root = os.path.dirname(os.getcwd())
sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, get_peft_model, LoraConfig

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 1. Understanding SafeLoRA Projection

SafeLoRA works by projecting LoRA weights onto a safety-aligned subspace. The key insight:

- **V** = W_aligned - W_base (safety direction vector)
- **P** = V @ V.T / ||V||² (projection matrix)
- **ΔW_safe** = P @ ΔW (projected LoRA update)


In [ ]:
# Demonstration of SafeLoRA projection concept (2D visualization)
np.random.seed(42)

# Simulate safety direction (difference between aligned and base model)
safety_direction = np.array([0.8, 0.6])
safety_direction = safety_direction / np.linalg.norm(safety_direction)

# Simulate LoRA update (potentially unsafe)
lora_update = np.array([0.5, -0.7])

# Compute projection onto safety subspace
projection_matrix = np.outer(safety_direction, safety_direction)
projected_update = projection_matrix @ lora_update

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))
origin = np.array([0, 0])

ax.quiver(*origin, *safety_direction, angles='xy', scale_units='xy', scale=1,
          color='green', width=0.02, label='Safety Direction (V)')
ax.quiver(*origin, *lora_update, angles='xy', scale_units='xy', scale=1,
          color='red', width=0.02, label='Original LoRA Update (ΔW)')
ax.quiver(*origin, *projected_update, angles='xy', scale_units='xy', scale=1,
          color='blue', width=0.02, label='Projected Update (P·ΔW)')

ax.plot([lora_update[0], projected_update[0]], [lora_update[1], projected_update[1]],
        'k--', alpha=0.5, linewidth=1.5, label='Projection')

ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5)
ax.legend(loc='upper left', fontsize=10)
ax.set_title('SafeLoRA Projection Visualization', fontsize=14, fontweight='bold')
ax.set_xlabel('Dimension 1', fontsize=12)
ax.set_ylabel('Dimension 2', fontsize=12)

cos_sim = np.dot(lora_update, safety_direction) / (np.linalg.norm(lora_update) * np.linalg.norm(safety_direction))
ax.text(0.05, -1.1, f'Cosine Similarity: {cos_sim:.3f}', fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nOriginal LoRA update: {lora_update}")
print(f"Projected update: {projected_update}")
print(f"Cosine similarity to safety direction: {cos_sim:.4f}")


## 2. SafeLoRA Configuration for Quantized Models


In [ ]:
# Using the centralized Pydantic configuration
from src.experiment_config import (
    BASELINE_CONFIG, 
    SAFELORA_CONFIG,
    create_baseline_config,
    create_safelora_config,
)

# View default configurations
print("=" * 50)
print("DEFAULT BASELINE CONFIGURATION")
print("=" * 50)
print(f"Base model: {BASELINE_CONFIG.model.base_model}")
print(f"PEFT model: {BASELINE_CONFIG.model.peft_model}")
print(f"4-bit: {BASELINE_CONFIG.quantization.load_in_4bit}")
print(f"Harmful prompts: {BASELINE_CONFIG.evaluation.harmful_prompts_path}")
print(f"Num samples: {BASELINE_CONFIG.evaluation.num_samples}")

print("\n" + "=" * 50)
print("DEFAULT SAFELORA CONFIGURATION")
print("=" * 50)
print(f"Base model: {SAFELORA_CONFIG.model.base_model}")
print(f"Aligned model: {SAFELORA_CONFIG.model.aligned_model}")
print(f"PEFT model: {SAFELORA_CONFIG.model.peft_model}")
print(f"Layer selection: {SAFELORA_CONFIG.projection.select_layers_type}")
print(f"Num proj layers: {SAFELORA_CONFIG.projection.num_proj_layers}")
print(f"Projection strength: {SAFELORA_CONFIG.projection.projection_strength}")

# Create custom configuration using helper function
custom_config = create_safelora_config(
    base_model="meta-llama/Llama-2-7b-hf",
    aligned_model="meta-llama/Llama-2-7b-chat-hf",
    num_proj_layers=15,
    num_samples=50,
)
print("\n" + "=" * 50)
print("CUSTOM CONFIGURATION EXAMPLE")
print("=" * 50)
print(f"Num proj layers: {custom_config.projection.num_proj_layers}")
print(f"Num samples: {custom_config.evaluation.num_samples}")


## 3. Full SafeLoRA Pipeline Example

Uncomment the code below to run with actual models (requires GPU with sufficient memory).


In [ ]:
# Full pipeline example using centralized configuration
# (uncomment to run with actual models)

"""
from experiments.run_baseline import run_baseline
from experiments.run_safelora import run_safelora
from src.experiment_config import create_baseline_config, create_safelora_config

# ============================================
# OPTION 1: Run with default configs
# (Modify BASELINE_CONFIG and SAFELORA_CONFIG in src/experiment_config.py)
# ============================================

# Run baseline
# from experiments.run_baseline import main as run_baseline_main
# run_baseline_main()

# Run SafeLoRA
# from experiments.run_safelora import main as run_safelora_main
# run_safelora_main()


# ============================================
# OPTION 2: Run with custom configs in code
# ============================================

# Create custom baseline config
baseline_config = create_baseline_config(
    base_model="meta-llama/Llama-2-7b-chat-hf",
    peft_model="path/to/your/lora/adapter",  # Update this!
    harmful_prompts_path="data/advbench_harmful_behaviors.csv",
    num_samples=100,
    load_in_4bit=True,
    output_dir="results/baseline",
)

# Run baseline evaluation
# run_baseline(baseline_config)

# Create custom SafeLoRA config
safelora_config = create_safelora_config(
    base_model="meta-llama/Llama-2-7b-hf",
    aligned_model="meta-llama/Llama-2-7b-chat-hf",
    peft_model="path/to/your/lora/adapter",  # Update this!
    num_proj_layers=10,
    projection_strength=1.0,
    harmful_prompts_path="data/advbench_harmful_behaviors.csv",
    num_samples=100,
    output_dir="results/safelora",
)

# Run SafeLoRA evaluation
# run_safelora(safelora_config)
"""

print("Full pipeline code is commented out.")
print("\\nTo run experiments:")
print("  1. Modify BASELINE_CONFIG and SAFELORA_CONFIG in src/experiment_config.py")
print("  2. Run: python experiments/run_baseline.py")
print("  3. Run: python experiments/run_safelora.py")
print("  4. Run: python experiments/compare_results.py")


## Next Steps

1. **Data Preparation**: Obtain harmful behavior datasets (e.g., AdvBench from your TML/ADV-LLM folder)
2. **Model Fine-tuning**: Create a fine-tuned LoRA model that exhibits safety degradation
3. **Baseline Evaluation**: Run `experiments/run_baseline.py` to measure initial safety
4. **SafeLoRA Evaluation**: Run `experiments/run_safelora.py` to measure improved safety
5. **Analysis**: Use `experiments/compare_results.py` to analyze improvements

### Research Questions to Explore:
- Does quantization itself (before fine-tuning) affect safety alignment?
- Is it better to project more layers with lower strength, or fewer layers with full strength?
- How does SafeLoRA perform with GPTQ vs AWQ vs bitsandbytes?
- How much does SafeLoRA projection affect downstream task performance?
